# Experiment 1: summarize one approach
CPU-only analysis of completed recordings. Edit the configuration below, then Run All. A single checkpoint is required; mixed protocols/checkpoints fail validation. Set `ignore_evaluator_source_hash=True` only to waive evaluator-source fingerprint checks; all other compatibility checks remain strict. Use the same flag for all approach summaries you intend to compare. Evaluation seeds are not training seeds.

In [7]:
import os, json, sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Find this checkout whether Jupyter starts in the repository or notebooks directory.
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebooks" / "exp1_analysis.py").exists())
sys.path.insert(0, str(REPO / "notebooks"))
import exp1_analysis as exp1
# Refresh helper changes even when this Jupyter kernel was already running.
import importlib
exp1 = importlib.reload(exp1)

CONFIG = {
    "input_roots": [str(REPO / "evaluation/exp1/b1z1_pact_ab4_coupled_none")],
    "task": "b1z1_pact_ab4_coupled_none",
    "checkpoint_sha256": None,  # Optional exact hash from metadata.json; otherwise require a unique hash.
    "ignore_evaluator_source_hash": True,  # Opt in to ignore evaluator-code hashes ONLY; settings/checkpoints remain strict.
    "display_name": "AB4",
    "expected_seeds": [0, 1, 2],
    "output_dir": str(REPO / "analysis/exp1/b1z1_pact_ab4_coupled_none"),
    "histogram_edges": np.linspace(-0.5, 0.5, 101).tolist(),  # Metres; reuse across approaches.
    "histogram_frame": "yaw",  # Successor base-yaw frame; "world" is also supported.
    "include_combined": True,  # Explicitly opt in to pooled stationary+locomotion trials.
    "acceleration_metrics": True,  # Backward differences, no filtering, no reset/gap crossings.
}
# Optional JSON override for reproducible batch execution/testing.
if os.environ.get("EXP1_ANALYSIS_CONFIG"):
    CONFIG.update(json.loads(Path(os.environ["EXP1_ANALYSIS_CONFIG"]).read_text()))
CONFIG


{'input_roots': ['/home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/evaluation/exp1/b1z1_pact_ab4_coupled_none'],
 'task': 'b1z1_pact_ab4_coupled_none',
 'checkpoint_sha256': None,
 'ignore_evaluator_source_hash': True,
 'display_name': 'AB4',
 'expected_seeds': [0, 1, 2],
 'output_dir': '/home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/analysis/exp1/b1z1_pact_ab4_coupled_none',
 'histogram_edges': [-0.5,
  -0.49,
  -0.48,
  -0.47,
  -0.46,
  -0.45,
  -0.44,
  -0.43,
  -0.42,
  -0.41000000000000003,
  -0.4,
  -0.39,
  -0.38,
  -0.37,
  -0.36,
  -0.35,
  -0.33999999999999997,
  -0.32999999999999996,
  -0.32,
  -0.31,
  -0.3,
  -0.29000000000000004,
  -0.28,
  -0.27,
  -0.26,
  -0.25,
  -0.24,
  -0.22999999999999998,
  -0.21999999999999997,
  -0.21000000000000002,
  -0.2,
  -0.19,
  -0.18,
  -0.16999999999999998,
  -0.15999999999999998,
  -0.14999999999999997,
  -0.14,
  -0.13,
  -0.12,
  -0.10999999999999999,
  -0.09999999999999998,
 

## Definitions
Event recovery is also evaluated separately for EE-position tolerance and for planar velocity plus yaw-rate tolerances. The eligible-event denominator, response window, contiguous dwell and whole-trial failure censoring are unchanged. Per-event outcomes and per-seed recovered counts are saved; each seed's recovery fraction is averaged equally across seeds. Regenerate summaries to add these metrics.
The additional `S_track_velocity` requires planar velocity **and** yaw-rate tolerances, while `S_track_manipulation` requires only the EE-position tolerance. Each uses the full prescribed horizon, with zero credit for failed/missing steps and only its own errors required finite. These are computed from transitions, then averaged equally across trials and evaluation seeds. Rerun this notebook for all methods before comparing the new scores.
Tracking errors use finite pre-failure samples. Each trial contributes equally within its evaluation seed; seeds then contribute equally. `S_track` uses the full prescribed horizon, including zero credit after failure. SD is the sample SD (`ddof=1`); one seed has undefined SD.

Physical-limit percentages pool valid joint-substep numerators/denominators within each seed. Event peaks and integrals require complete response windows. Any physical failure censors all recovery events in that trial, matching the evaluator. Recovery times are averaged only over recovered events; no missing value becomes zero.

In [8]:
summary = exp1.summarize(CONFIG)
display(pd.DataFrame(summary["aggregates"]))
print("\n".join(summary["diagnostics"]) or "No missing-data or online cross-check warnings.")
print("Exported:", CONFIG["output_dir"])


/tmp/ipykernel_117985/2162493752.py:1: RuntimeWarning: Evaluator-source hash validation disabled by ignore_evaluator_source_hash=True; evaluation-code equivalence has not been verified. Original hashes are retained in provenance.
  summary = exp1.summarize(CONFIG)


,condition,load_range,motion,metric,unit,mean,sd,contributing_seeds,attempted_trials,contributing_trials
0,combined-disturbance,in,combined,ee_error_m,m,0.068286,0.001627,3,300,300.0
1,combined-disturbance,in,combined,planar_velocity_error_m_s,m/s,0.077197,0.001533,3,300,300.0
2,combined-disturbance,in,combined,yaw_error_rad_s,rad/s,0.097684,0.003213,3,300,300.0
3,combined-disturbance,in,combined,S_track,fraction,0.758206,0.020208,3,300,300.0
4,combined-disturbance,in,combined,failure_rate,fraction,0.006667,0.005774,3,300,300.0
...,...,...,...,...,...,...,...,...,...,...
157,nominal,in,stationary,recovery_fraction,fraction,NaN,NaN,0,150,NaN
158,nominal,in,stationary,recovery_fraction_ee,fraction,NaN,NaN,0,150,NaN
159,nominal,in,stationary,recovery_fraction_velocity,fraction,NaN,NaN,0,150,NaN
160,nominal,in,stationary,torso_angular_acceleration_rms,rad/s^2,7.086920,0.041270,3,150,NaN


Evaluator-source hash validation disabled by ignore_evaluator_source_hash=True; evaluation-code equivalence has not been verified. Original hashes are retained in provenance.
Online trial mismatch: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/evaluation/exp1/b1z1_pact_ab4_coupled_none/combine_in/seed_2, trial 78, S_track
Online seed mismatch: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/evaluation/exp1/b1z1_pact_ab4_coupled_none/combine_in/seed_2, stationary, S_track
Exported: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/analysis/exp1/b1z1_pact_ab4_coupled_none


## Portable exports
Exact equal-seed weighted box statistics (5/25/50/75/95 percentiles) are also exported in `summary.json`. They include all valid errors, including histogram tails. Rerun this notebook for older summaries that lack `ee_boxplots`.
`summary.json` contains provenance, protocols, per-trial/event results, per-seed values, aggregate statistics and diagnostics. `summary.csv` contains aggregate rows; `ee_histograms.csv` contains per-seed counts and explicit underflow/overflow denominators. `COMPLETE` is written only after all exports succeed.

Histograms use signed measured-minus-reference errors, in metres, rotated with the matching successor orientation. Counts are normalized per seed using **all valid samples**, including tails, before seed averaging. Failed tails are never padded.

In [9]:
histograms = pd.read_csv(Path(CONFIG["output_dir"]) / "ee_histograms.csv")
tails = histograms.drop_duplicates(["seed", "condition", "load_range", "motion", "axis"])
display(tails[["seed", "condition", "load_range", "motion", "axis", "valid_samples", "underflow", "overflow"]])


,seed,condition,load_range,motion,axis,valid_samples,underflow,overflow
0,0,combined-disturbance,in,stationary,x,30000,0,0
100,0,combined-disturbance,in,stationary,y,30000,0,0
200,0,combined-disturbance,in,stationary,z,30000,0,0
300,0,combined-disturbance,in,locomotion,x,30000,2,0
400,0,combined-disturbance,in,locomotion,y,30000,0,0
500,0,combined-disturbance,in,locomotion,z,30000,0,0
600,0,combined-disturbance,in,combined,x,60000,2,0
700,0,combined-disturbance,in,combined,y,60000,0,0
800,0,combined-disturbance,in,combined,z,60000,0,0
900,1,combined-disturbance,in,stationary,x,30000,0,0
